In [1]:
# ============================================================================
# APOLLO VIT ABLATION NOTEBOOK – DeiT-Small on CIFAR-100
# (All reviewer comments addressed; Kaggle Offline Ready)
#
# QWEN & EXPERT FIXES APPLIED:
# 1. FINE-TUNING STRATEGY: Using Pre-trained DeiT-Small (22M params)
#    with 224x224 resizing for ultra-fast, single-session execution on Kaggle.
# 2. Fixed Fatal Statistical Paradox (d_z vs p-value) using Paired T-Test.
# 3. Cleaned Metrics to standard (Acc, F1, AUC, MCC, Time).
# 4. Added Pure_AdamW/Pure_Lion baselines for complete ablation.
# 5. Kaggle Smart Checkpointing and Bulletproof Offline Loading included.
# 6. Fixed standard Lion update formula inside Apollo.
# 7. Auto-ZIP Export enabled for LaTeX tables and plots.
# ============================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import os, time, warnings, json, random, pickle, shutil, math, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.optimizer import Optimizer
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
from PIL import Image
from sklearn.metrics import (f1_score, roc_auc_score, matthews_corrcoef, confusion_matrix)
from scipy.stats import ttest_rel, t as t_dist
from statsmodels.stats.multitest import multipletests

# Auto-install and import timm for lightweight ViT/DeiT architectures
try:
    import timm
except ImportError:
    os.system('pip install timm')
    import timm

# ============================================================================
# 🛡️ ENVIRONMENT & REPRODUCIBILITY [R2 #3]
# ============================================================================
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
warnings.filterwarnings("ignore")
try:
    torch.set_float32_matmul_precision('high')
except:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Execution Device Verified: {device}", flush=True)

# Prevent hidden C++ SegFaults on Kaggle GPUs
torch.backends.cudnn.benchmark = False 

# Increased to 3 seeds for Statistical Power
ABLATION_SEEDS = [42, 123, 777]  
SENSITIVITY_SEED = 42            

# EXACT KAGGLE PATHS (Discovered via OS Scan)
CIFAR100_SRC = '/kaggle/input/datasets/shadowhexer/cifar-100'
TIMM_WEIGHTS_PATH = '/kaggle/input/datasets/kami2suukyi/timm-pretrained-vit/vit/deit_small_patch16_224-cd65a155.pth'

# Output directory
RESULTS_DIR = "/kaggle/working/results_apollo_vit_ablation"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "latex_tables"), exist_ok=True)

# Smart Checkpoint File
CHECKPOINT_FILE = os.path.join(RESULTS_DIR, "vit_ablation_checkpoint.json")

# Global execution timer
GLOBAL_START_TIME = time.time()
SAFE_TIME_LIMIT_SECONDS = 11.4 * 3600  

# ============================================================================
# ⚙️ CONFIGURATION
# ============================================================================
VIT_EPOCHS = 9
BATCH_SIZE = 64             
GRADIENT_CLIP_VALUE = 1.0
WARMUP_EPOCHS = 2            

# ============================================================================
# 🔄 REPRODUCIBILITY
# ============================================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    def seed_worker(worker_id):
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed)
        random.seed(worker_seed)
    return seed_worker

# ============================================================================
# 🧠 APOLLO OPTIMIZER
# ============================================================================
class Apollo(Optimizer):
    def __init__(self, params, lr=3e-4, betas=(0.9, 0.999, 0.9), eps=1e-8,
                 weight_decay=0.05, mode='standard', total_steps=None):
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
        self.mode = mode
        self.total_steps = total_steps
        self.current_step = 0
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        self.current_step += 1
        for group in self.param_groups:
            beta1, beta2, beta3 = group['betas']
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)
                    state['update_agreement'] = torch.zeros((), device=p.device)
                    state['last_cs'] = 0.0
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                state['step'] += 1

                if group['weight_decay'] != 0:
                    p.add_(p, alpha=-group['weight_decay'] * group['lr'])

                lion_c = exp_avg.mul(beta1).add(grad, alpha=1 - beta1)
                u_lion = torch.sign(lion_c)

                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                bc1 = 1 - beta1 ** state['step']
                bc2 = 1 - beta2 ** state['step']
                m_hat = exp_avg / bc1
                v_hat = exp_avg_sq / bc2
                denom = v_hat.sqrt().add_(group['eps'])
                u_adam = m_hat / denom          

                adam_norm = u_adam.norm(p=2)
                lion_norm = u_lion.norm(p=2)
                if lion_norm > 0:
                    u_lion = u_lion * (adam_norm / lion_norm)

                cs_raw = F.cosine_similarity(u_adam.flatten(), u_lion.flatten(), dim=0, eps=1e-10)
                state['last_cs'] = cs_raw.item()

                if self.mode == 'pure_adamw':
                    gamma = 0.0
                elif self.mode == 'pure_lion':
                    gamma = 1.0
                elif self.mode == 'linear':
                    if self.total_steps is not None:
                        progress = self.current_step / self.total_steps
                        gamma = max(0.0, min(1.0, progress))
                    else:
                        gamma = 0.5
                elif self.mode == 'no_cosine':
                    gamma = 0.5
                else:  
                    state['update_agreement'].mul_(beta3).add_(cs_raw, alpha=1 - beta3)
                    bc3 = 1 - beta3 ** state['step']
                    agreement_hat = state['update_agreement'] / bc3
                    gamma = (agreement_hat + 1.0) / 2.0

                p.add_((1 - gamma) * u_adam + gamma * u_lion, alpha=-group['lr'])
        return loss

# ============================================================================
# 📂 CIFAR-100 OFFLINE DATASET 
# ============================================================================
class CIFAR100Manual(Dataset):
    def __init__(self, root, train=True, transform=None):
        self.root = root
        self.transform = transform
        base_dir = os.path.join(root, 'cifar-100-python')
        filename = 'train' if train else 'test'
        with open(os.path.join(base_dir, filename), 'rb') as f:
            batch = pickle.load(f, encoding='latin1')
        self.data = batch['data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        self.targets = batch['fine_labels']

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img = Image.fromarray(self.data[idx])
        target = int(self.targets[idx])
        if self.transform:
            img = self.transform(img)
        return img, target

def prepare_offline_cifar100():
    target_base = '/kaggle/working/cifar100_data'
    if os.path.exists(os.path.join(target_base, 'cifar-100-python', 'train')):
        return target_base
    src_dir = os.path.join(CIFAR100_SRC, 'cifar-100-python')
    if not os.path.exists(src_dir):
        src_dir = CIFAR100_SRC 
    target_sub = os.path.join(target_base, 'cifar-100-python')
    os.makedirs(target_base, exist_ok=True)
    shutil.copytree(src_dir, target_sub, dirs_exist_ok=True)
    return target_base

def get_cifar100_loaders(batch_size=128, seed=42):
    data_root = prepare_offline_cifar100()
    
    transform_train = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2761)),
        transforms.RandomErasing(p=0.25)
    ])
    transform_test = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2761))
    ])

    full_train = CIFAR100Manual(data_root, train=True, transform=None)
    test_ds = CIFAR100Manual(data_root, train=False, transform=transform_test)

    gen = torch.Generator().manual_seed(seed)
    n_train = int(0.9 * len(full_train))
    n_val = len(full_train) - n_train
    train_sub, val_sub = random_split(full_train, [n_train, n_val], generator=gen)

    class SubsetWithTransform(Dataset):
        def __init__(self, dataset, indices, transform=None):
            self.dataset = dataset
            self.indices = indices
            self.transform = transform
        def __len__(self):
            return len(self.indices)
        def __getitem__(self, idx):
            img, target = self.dataset[self.indices[idx]]
            if self.transform:
                img = self.transform(img)
            return img, target

    train_ds = SubsetWithTransform(full_train, train_sub.indices, transform_train)
    val_ds = SubsetWithTransform(full_train, val_sub.indices, transform_test)

    seed_worker = set_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=4, pin_memory=True, worker_init_fn=seed_worker)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    return train_loader, val_loader, test_loader

# ============================================================================
# 🧠 VISION TRANSFORMER MODEL (DeiT-Small) - OFFLINE SAFE
# ============================================================================
class ViT_CIFAR100(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.model = timm.create_model('deit_small_patch16_224', pretrained=False, num_classes=num_classes)
        
        weights_loaded = False
        
        if os.path.exists(TIMM_WEIGHTS_PATH):
            try:
                state_dict = torch.load(TIMM_WEIGHTS_PATH, map_location='cpu')
                if 'model' in state_dict: state_dict = state_dict['model']
                state_dict = {k: v for k, v in state_dict.items() if not k.startswith('head')}
                self.model.load_state_dict(state_dict, strict=False)
                print(f"      [Model] ✅ EXACT path matched! Offline weights loaded from: {TIMM_WEIGHTS_PATH}", flush=True)
                weights_loaded = True
            except Exception as e:
                print(f"      [Model] ⚠️ Error loading exact path: {e}", flush=True)
        
        if not weights_loaded:
            print("      [Model] 🔍 Exact path missed. Scanning all Kaggle directories...", flush=True)
            for root, dirs, files in os.walk('/kaggle/input'):
                for file in files:
                    if 'deit_small_patch16_224' in file and file.endswith('.pth'):
                        target = os.path.join(root, file)
                        try:
                            state_dict = torch.load(target, map_location='cpu')
                            if 'model' in state_dict: state_dict = state_dict['model']
                            state_dict = {k: v for k, v in state_dict.items() if not k.startswith('head')}
                            self.model.load_state_dict(state_dict, strict=False)
                            print(f"      [Model] ✅ SUCCESS! Weights dynamically found at: {target}", flush=True)
                            weights_loaded = True
                            break
                        except: pass
                if weights_loaded: break
        
        if not weights_loaded:
            print("      [Model] ❌ CRITICAL: Weights NOT FOUND anywhere. Using random init!", flush=True)

    def forward(self, x):
        return self.model(x)

# ============================================================================
# 🔄 TRAINING LOOP
# ============================================================================
def train_and_eval(model, train_loader, val_loader, test_loader, optimizer,
                   total_steps, warmup_steps, num_epochs, use_amp=True):
    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler() if use_amp else None

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(1e-4, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_val_loss = float('inf')
    best_state = None
    no_improve = 0
    patience = 5

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'epoch_time': []}
    raw_cosine_hist = []
    gamma_hist = []

    for epoch in range(num_epochs):
        if time.time() - GLOBAL_START_TIME > SAFE_TIME_LIMIT_SECONDS:
            print(f"\n⚠️ ⏳ [GRACEFUL EXIT] Runtime threshold reached at Epoch {epoch+1}. Halting safely...", flush=True)
            return None 
            
        t0 = time.time()
        model.train()
        tr_loss, tr_correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad()
            if use_amp:
                with torch.cuda.amp.autocast():
                    out = model(x)
                    loss = criterion(out, y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
                scaler.step(optimizer)
                scaler.update()
            else:
                out = model(x)
                loss = criterion(out, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
                optimizer.step()
            scheduler.step()
            tr_loss += loss.item() * x.size(0)
            tr_correct += out.argmax(1).eq(y).sum().item()
            total += y.size(0)

        epoch_time = time.time() - t0
        history['epoch_time'].append(epoch_time)
        history['train_loss'].append(tr_loss / total)
        history['train_acc'].append(tr_correct / total)

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                out = model(x)
                loss = criterion(out, y)
                val_loss += loss.item() * x.size(0)
                val_correct += out.argmax(1).eq(y).sum().item()
                val_total += y.size(0)
        history['val_loss'].append(val_loss / val_total)
        history['val_acc'].append(val_correct / val_total)

        if isinstance(optimizer, Apollo):
            if optimizer.param_groups and optimizer.param_groups[0]['params']:
                p0 = optimizer.param_groups[0]['params'][0]
                state = optimizer.state.get(p0, {})
                if 'last_cs' in state:
                    raw_cosine_hist.append(state['last_cs'])
                if optimizer.mode == 'standard':
                    if 'update_agreement' in state:
                        ag = state['update_agreement'].item()
                        gamma = (ag + 1) / 2.0
                        gamma_hist.append(gamma)
                    else:
                        gamma_hist.append(0.5)
                elif optimizer.mode == 'linear':
                    gamma = min(1.0, optimizer.current_step / optimizer.total_steps)
                    gamma_hist.append(gamma)
                elif optimizer.mode == 'pure_adamw':
                    gamma_hist.append(0.0)
                elif optimizer.mode == 'pure_lion':
                    gamma_hist.append(1.0)
                else:
                    gamma_hist.append(0.5)
        else:
            raw_cosine_hist.append(0.0)
            gamma_hist.append(0.0)

        if history['val_loss'][-1] < best_val_loss - 1e-4:
            best_val_loss = history['val_loss'][-1]
            no_improve = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            
        print(f"      🔹 Epoch {epoch+1}/{num_epochs} | Val Acc: {history['val_acc'][-1]:.4f} | Time: {epoch_time:.0f}s", flush=True)
        
        if no_improve >= patience and epoch >= patience:
            if best_state:
                model.load_state_dict(best_state)
            break

    if best_state:
        model.load_state_dict(best_state)
        
    model.eval()
    all_preds, all_targets, all_probs = [], [], []
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            out = model(x)
            probs = F.softmax(out, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_preds.extend(out.argmax(1).cpu().numpy())
            all_targets.extend(y.cpu().numpy())
            
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    all_probs = np.vstack(all_probs)

    acc = float((all_preds == all_targets).mean())
    f1 = float(f1_score(all_targets, all_preds, average='macro', zero_division=0))
    try: auc_val = float(roc_auc_score(all_targets, all_probs, multi_class='ovr', average='macro'))
    except: auc_val = 0.0
    mcc = float(matthews_corrcoef(all_targets, all_preds))
    avg_epoch_time = float(np.mean(history['epoch_time']))

    return {
        'test_acc': acc, 'test_f1': f1, 'test_auc': auc_val, 'test_mcc': mcc,
        'avg_epoch_time': avg_epoch_time, 'history': history,
        'raw_cosine_hist': raw_cosine_hist, 'gamma_hist': gamma_hist
    }

# ============================================================================
# 💾 CHECKPOINTING UTILS
# ============================================================================
def load_json(filepath):
    if os.path.exists(filepath):
        try:
            with open(filepath, 'r') as f: return json.load(f)
        except json.JSONDecodeError: pass
    return {}

def save_json(data, filepath):
    with open(filepath, 'w') as f: json.dump(data, f, indent=4)

# ============================================================================
# 📊 STATISTICAL ANALYSIS
# ============================================================================
def adaptive_ci(data, cl=0.95):
    n = len(data)
    if n < 5:
        mean, std = np.mean(data), np.std(data, ddof=1) if n > 1 else 0.0
        t_val = t_dist.ppf((1 + cl) / 2, max(n - 1, 1))
        margin = t_val * std / np.sqrt(n) if n > 0 else 0
        return float(mean - margin), float(mean + margin)
    rng = np.random.default_rng(42)
    boot = [np.mean(rng.choice(data, n, replace=True)) for _ in range(1000)]
    alpha = (1 - cl) / 2
    return float(np.percentile(boot, 100 * alpha)), float(np.percentile(boot, 100 * (1 - alpha)))

def compute_statistics(multi_results):
    metrics = ['test_acc', 'test_f1', 'test_auc', 'test_mcc', 'avg_epoch_time']
    agg, var, cvs, ci = {}, {}, {}, {}
    for opt, runs in multi_results.items():
        if not runs: continue
        agg[opt], var[opt], cvs[opt], ci[opt] = {}, {}, {}, {}
        for m in metrics:
            vals = [r[m] for r in runs]
            mean_v = float(np.mean(vals))
            std_v = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            agg[opt][f'{m}_mean'] = mean_v
            agg[opt][f'{m}_std'] = std_v
            var[opt][f'{m}_var'] = float(np.var(vals, ddof=1)) if len(vals) > 1 else 0.0
            cvs[opt][f'{m}_cv'] = float((std_v / mean_v * 100) if mean_v > 0 else 0)
            ci[opt][m] = adaptive_ci(vals)
    return agg, var, cvs, ci

def statistical_tests(results, baseline='Apollo_β3=0.9'):
    if baseline not in results or len(results[baseline]) < 2: return {}
    base_vals = np.array([r['test_acc'] for r in results[baseline]])
    sig = {}; pvals = []; basenames = []
    
    for opt, runs in results.items():
        if opt == baseline or not runs or len(runs) < 2: continue
        vals = np.array([r['test_acc'] for r in runs])
        if len(base_vals) != len(vals): continue
        basenames.append(opt)
        
        diffs = base_vals - vals
        mean_diff = float(np.mean(diffs))
        std_diff = float(np.std(diffs, ddof=1))
        
        if std_diff < 1e-8:
            d_z = 0.0
            p_raw = 1.0 if abs(mean_diff) < 1e-8 else 0.0
        else:
            d_z = float(mean_diff / std_diff)
            try: _, p_raw = ttest_rel(base_vals, vals)
            except: p_raw = 1.0
            
        pvals.append(float(p_raw))
        sig[opt] = {'diff': mean_diff, 'p_raw': float(p_raw), 'es_z': d_z}
        
    if pvals:
        _, p_corr, _, _ = multipletests(pvals, method='fdr_bh')
        for i, opt in enumerate(basenames):
            sig[opt]['p_corr'] = float(p_corr[i])
            sig[opt]['sig'] = '***' if p_corr[i] < 0.001 else '**' if p_corr[i] < 0.01 else '*' if p_corr[i] < 0.05 else 'ns'
    return sig

# ============================================================================
# 📄 LATEX TABLE GENERATORS 
# ============================================================================
def ablation_performance_table(agg, ci, caption, label, filename):
    mc = [{'d':'Acc','k':'test_acc','f':'.4f'}, {'d':'F1','k':'test_f1','f':'.4f'},
          {'d':'AUC','k':'test_auc','f':'.4f'}, {'d':'MCC','k':'test_mcc','f':'.4f'},
          {'d':'Time(s)','k':'avg_epoch_time','f':'.2f'}]
    lines = [
        r"\begin{table}[htbp]", r"\centering", r"\caption{" + caption + "}",
        r"\label{" + label + "}", r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{l" + "c" * len(mc) + "}", r"\toprule",
        r"{\bf Config} & " + " & ".join([f"{{\\bf {m['d']}}}" for m in mc]) + r" \\", r"\midrule"
    ]
    order = ['Pure_AdamW', 'Pure_Lion', 'Apollo_β3=0', 'Apollo_β3=0.9', 'Apollo_β3=0.99', 'Apollo_Fixedγ', 'Apollo_Linear']
    for opt in order:
        if opt not in agg: continue
        row = [opt.replace('_', ' ')]
        for m in mc:
            k, f = m['k'], m['f']
            mean, std = agg[opt][f'{k}_mean'], agg[opt][f'{k}_std']
            low, high = ci[opt][k]
            row.append(f"{mean:{f}} $\\pm$ {std:{f}} \\; [{low:{f}}, {high:{f}}]")
        lines.append(" & ".join(row) + r" \\")
    lines.extend([r"\bottomrule", r"\end{tabular}", r"}", r"\end{table}"])
    with open(os.path.join(RESULTS_DIR, "latex_tables", filename), "w") as f: f.write("\n".join(lines))

def stats_table(sig, caption, label, filename):
    lines = [
        r"\begin{table}[htbp]", r"\centering", r"\caption{" + caption + "}",
        r"\label{" + label + "}", r"\begin{tabular}{lccccc}", r"\toprule",
        r"\textbf{Comparison vs $\beta_3$=0.9} & \textbf{$\Delta$ Acc} & \textbf{$p_{\text{ttest}}$} & \textbf{$p_{\text{FDR}}$} & \textbf{$d_z$} & \textbf{Sig.} \\",
        r"\midrule"
    ]
    for opt, r in sig.items():
        diff_str = f"${r['diff']:+.4f}$" if abs(r['diff']) >= 1e-4 else f"${r['diff']:+.1e}$"
        lines.append(f"{opt.replace('_',' ')} & {diff_str} & {r['p_raw']:.3f} & {r['p_corr']:.3f} & {r['es_z']:.3f} & {r['sig']} \\\\")
    lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
    with open(os.path.join(RESULTS_DIR, "latex_tables", filename), "w") as f: f.write("\n".join(lines))

def variance_table(var, cvs, caption, label, filename):
    metrics_show = [('test_acc', 'Accuracy'), ('test_f1', 'F1'), ('test_auc', 'AUC')]
    lines = [
        r"\begin{table}[htbp]", r"\centering", r"\caption{" + caption + "}",
        r"\label{" + label + "}", r"\begin{tabular}{l" + "cc" * len(metrics_show) + "}", r"\toprule",
        r"\multirow{2}{*}{\bf Config} " + "".join([f"& \\multicolumn{{2}}{{c}}{{\\bf {disp}}}" for _, disp in metrics_show]) + r" \\",
        r"\cmidrule(lr){2-3} \cmidrule(lr){4-5} \cmidrule(lr){6-7}",
        r" & " + " & ".join([r"$\sigma^2$ & CV\%"] * len(metrics_show)) + r" \\", r"\midrule"
    ]
    order = ['Pure_AdamW', 'Pure_Lion', 'Apollo_β3=0', 'Apollo_β3=0.9', 'Apollo_β3=0.99', 'Apollo_Fixedγ', 'Apollo_Linear']
    for opt in order:
        if opt not in var: continue
        row = [opt.replace('_', ' ')]
        for key, _ in metrics_show:
            v = var[opt][f'{key}_var']
            if v < 1e-6: row.append(r"$<10^{-6}$")
            else: row.append(f"{v:.2e}")
            row.append(f"{cvs[opt][f'{key}_cv']:.2f}\\%")
        lines.append(" & ".join(row) + r" \\")
    lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
    with open(os.path.join(RESULTS_DIR, "latex_tables", filename), "w") as f: f.write("\n".join(lines))

# ============================================================================
# 📈 PLOTTING FUNCTIONS
# ============================================================================
def plot_ablation_dynamics(ablation_results, title, save_name):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    colors = plt.cm.tab10.colors
    configs = list(ablation_results.keys())
    
    # 1. Validation Accuracy
    ax = axes[0]
    for i, cfg in enumerate(configs):
        runs = ablation_results[cfg]
        if not runs: continue
        accs = [r['history']['val_acc'] for r in runs]
        max_len = max(len(a) for a in accs)
        padded = [np.pad(a, (0, max_len - len(a)), constant_values=np.nan) for a in accs]
        mean_acc = np.nanmean(padded, axis=0)
        ax.plot(range(1, max_len+1), mean_acc, label=cfg.replace('_', ' '), color=colors[i % len(colors)], linewidth=2)
    ax.set_title('Validation Accuracy', fontweight='bold')
    ax.set_xlabel('Epochs')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend(fontsize=9)

    # 2. Gamma Weight
    ax = axes[1]
    for i, cfg in enumerate(configs):
        runs = ablation_results[cfg]
        if not runs: continue
        gammas = [r['gamma_hist'] for r in runs]
        max_len = max(len(g) for g in gammas)
        padded = [np.pad(g, (0, max_len - len(g)), constant_values=np.nan) for g in gammas]
        mean_g = np.nanmean(padded, axis=0)
        ax.plot(range(1, max_len+1), mean_g, label=cfg.replace('_', ' '), color=colors[i % len(colors)], linewidth=2)
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.8)
    ax.set_title(r'Blending Weight $\gamma$', fontweight='bold')
    ax.set_xlabel('Epochs')
    ax.grid(True, linestyle='--', alpha=0.6)

    # 3. Raw Cosine Similarity
    ax = axes[2]
    for i, cfg in enumerate(configs):
        if 'Apollo' not in cfg and cfg != 'Pure_Lion': continue 
        runs = ablation_results[cfg]
        if not runs: continue
        cosines = [r['raw_cosine_hist'] for r in runs]
        if not any(cosines): continue
        max_len = max(len(c) for c in cosines if c)
        if max_len == 0: continue
        padded = [np.pad(c, (0, max_len - len(c)), constant_values=np.nan) for c in cosines if c]
        mean_c = np.nanmean(padded, axis=0)
        ax.plot(range(1, max_len+1), mean_c, label=cfg.replace('_', ' '), color=colors[i % len(colors)], alpha=0.8)
    ax.axhline(0.0, color='black', linestyle='-', alpha=0.5)
    ax.set_title('Raw Update Agreement (Cosine Similarity)', fontweight='bold')
    ax.set_xlabel('Epochs')
    ax.grid(True, linestyle='--', alpha=0.6)

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "plots", save_name), dpi=300, bbox_inches='tight')
    plt.close()

# ============================================================================
# 📦 FINAL ZIP EXPORT
# ============================================================================
def create_final_zip():
    zip_path = "/kaggle/working/vit_ablation_results.zip"
    base_folder = os.path.basename(RESULTS_DIR)
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        if os.path.exists(CHECKPOINT_FILE):
            zipf.write(CHECKPOINT_FILE, arcname=os.path.join(base_folder, os.path.basename(CHECKPOINT_FILE)))
        
        for sub_dir in ["plots", "latex_tables"]:
            dir_path = os.path.join(RESULTS_DIR, sub_dir)
            if os.path.isdir(dir_path):
                for root, dirs, files in os.walk(dir_path):
                    for file in files:
                        full_path = os.path.join(root, file)
                        arcname = os.path.join(base_folder, sub_dir, file)
                        zipf.write(full_path, arcname=arcname)
                        
    print(f"\n📦 Final results safely archived at: {zip_path}", flush=True)

# ============================================================================
# 🧪 ABLATION EXPERIMENTS WITH SMART CHECKPOINTING
# ============================================================================
def run_ablation_study():
    print("\n=== ViT ABLATION STUDY (DeiT-Small on CIFAR-100) ===")
    
    configs = {
        'Pure_AdamW': lambda p, ts: Apollo(p, lr=3e-4, weight_decay=0.05, mode='pure_adamw', total_steps=ts),
        'Pure_Lion': lambda p, ts: Apollo(p, lr=3e-4, weight_decay=0.05, mode='pure_lion', total_steps=ts),
        'Apollo_β3=0': lambda p, ts: Apollo(p, lr=3e-4, weight_decay=0.05, betas=(0.9,0.999,0.0), total_steps=ts),
        'Apollo_β3=0.9': lambda p, ts: Apollo(p, lr=3e-4, weight_decay=0.05, betas=(0.9,0.999,0.9), total_steps=ts),
        'Apollo_β3=0.99': lambda p, ts: Apollo(p, lr=3e-4, weight_decay=0.05, betas=(0.9,0.999,0.99), total_steps=ts),
        'Apollo_Fixedγ': lambda p, ts: Apollo(p, lr=3e-4, weight_decay=0.05, mode='no_cosine', total_steps=ts),
        'Apollo_Linear': lambda p, ts: Apollo(p, lr=3e-4, weight_decay=0.05, mode='linear', total_steps=ts)
    }
    
    results = load_json(CHECKPOINT_FILE)
    for name in configs:
        if name not in results: results[name] = []

    for seed in ABLATION_SEEDS:
        for cfg_name, cfg_fn in configs.items():
            if len([r for r in results[cfg_name] if r.get('seed') == seed]) > 0:
                print(f"Skipping {cfg_name} seed={seed} (Already completed)")
                continue

            set_seed(seed)
            train_loader, val_loader, test_loader = get_cifar100_loaders(BATCH_SIZE, seed)
            total_steps = VIT_EPOCHS * len(train_loader)
            warmup_steps = WARMUP_EPOCHS * len(train_loader)
            
            print(f"\n🔥 Running {cfg_name} with seed={seed}...")
            model = ViT_CIFAR100().to(device)
            optimizer = cfg_fn(model.parameters(), total_steps)
            
            res = train_and_eval(model, train_loader, val_loader, test_loader,
                                 optimizer, total_steps, warmup_steps, VIT_EPOCHS)
            
            if res is None:
                print("\n⚠️ ⏳ [GRACEFUL EXIT] Runtime budget expired. Building metrics tables cleanly...", flush=True)
                save_json(results, CHECKPOINT_FILE)
                return results

            res['seed'] = seed
            print(f"  ✔️ {cfg_name} seed={seed} Final Acc={res['test_acc']:.4f}")
            
            results[cfg_name].append(res)
            save_json(results, CHECKPOINT_FILE)
            
            # Memory cleanup
            del model, optimizer, res
            torch.cuda.empty_cache()
            
    return results

# ============================================================================
# 🚀 MAIN EXECUTION
# ============================================================================
if __name__ == "__main__":
    print("=" * 80)
    print("APOLLO ViT ABLATION NOTEBOOK – Kaggle Optimized & Fine-Tuned (DeiT-Small)")
    print("=" * 80)

    ablation_results = run_ablation_study()
    abl_agg, abl_var, abl_cvs, abl_ci = compute_statistics(ablation_results)
    
    if abl_agg:
        ablation_performance_table(abl_agg, abl_ci, r"ViT Ablation Study (DeiT-Small on CIFAR-100, Mean $\pm$ SD [95\% CI])",
                                    "tab:vit_ablation_perf", "t1_vit_ablation_perf.tex")
        
        abl_sig = statistical_tests(ablation_results, baseline='Apollo_β3=0.9')
        stats_table(abl_sig, r"Statistical Significance in ViT Ablation (vs $\beta_3$=0.9, Paired T-Test) - DeiT-Small on CIFAR-100",
                    "tab:vit_ablation_stats", "t2_vit_ablation_stats.tex")
                    
        variance_table(abl_var, abl_cvs, r"Stability Analysis (Variance \& CV) - DeiT-Small on CIFAR-100",
                       "tab:vit_ablation_var", "t3_vit_ablation_var.tex")
                       
        plot_ablation_dynamics(ablation_results, "ViT Ablation Dynamics (DeiT-Small on CIFAR-100)", "vit_ablation_dynamics.png")
        
        create_final_zip()
        print("\n✅ ViT ABLATION COMPLETE. Checkpoints, Results and LaTeX tables saved.")
    else:
        print("\n⚠️ Process interrupted before first completion. Run again to resume.")

🚀 Execution Device Verified: cuda
APOLLO ViT ABLATION NOTEBOOK – Kaggle Optimized & Fine-Tuned (DeiT-Small)

=== ViT ABLATION STUDY (DeiT-Small on CIFAR-100) ===

🔥 Running Pure_AdamW with seed=42...
      [Model] ✅ EXACT path matched! Offline weights loaded from: /kaggle/input/datasets/kami2suukyi/timm-pretrained-vit/vit/deit_small_patch16_224-cd65a155.pth
      🔹 Epoch 1/9 | Val Acc: 0.7428 | Time: 177s
      🔹 Epoch 2/9 | Val Acc: 0.7334 | Time: 178s
      🔹 Epoch 3/9 | Val Acc: 0.7724 | Time: 177s
      🔹 Epoch 4/9 | Val Acc: 0.7930 | Time: 177s
      🔹 Epoch 5/9 | Val Acc: 0.8074 | Time: 177s
      🔹 Epoch 6/9 | Val Acc: 0.8384 | Time: 176s
      🔹 Epoch 7/9 | Val Acc: 0.8512 | Time: 177s
      🔹 Epoch 8/9 | Val Acc: 0.8674 | Time: 176s
      🔹 Epoch 9/9 | Val Acc: 0.8728 | Time: 177s
  ✔️ Pure_AdamW seed=42 Final Acc=0.8662

🔥 Running Pure_Lion with seed=42...
      [Model] ✅ EXACT path matched! Offline weights loaded from: /kaggle/input/datasets/kami2suukyi/timm-pretrained-vit/v